## How we built the speculative decoding demo

The demo animates Figure 1 of Leviathan et al. (2023): a small model guesses tokens, the large model verifies them in one parallel run. Accepted guesses appear in green, the first rejection in red, and its correction in blue. We added one thing the paper's figure leaves out: the discarded guesses after the first rejection. They were generated, so we show them.

We built it in FastHTML inside this dialog. `JupyUvi` runs the server in a background thread and `HTMX()` shows the app in an iframe, so we could redefine routes and see changes without restarting anything.

The animation data is pre-scripted.

Key design decisions:

- **Server-driven steps over client-side JS.** Each step is an HTMX request that swaps in the next state. This mirrors the algorithm itself: one round-trip per run of the target model.
- **Stateless routes.** The tick number lives in the URL, not on the server. Any request fully describes what to render, which made the back button free.
- **One container for both tracks.** The standard and speculative tracks must advance on the same clock, so a single `/step` route renders both and swaps them together. Separate self-pulling rows could drift apart.
- **A shared clock of two ticks per model run.** Even ticks show the small model guessing, odd ticks show the verification. The standard track reveals one token per odd tick, at the same pace throughout. Keeping that pace matters: the payoff of the demo is watching the speculative track finish while the standard track is still three tokens in.
- **Buttons over a timer.** We started with timed self-advancing rows, then switched to Back and Next buttons stepping one tick at a time, so a viewer can sit on the verification beat as long as they like.
- **The paper's colors.** Black for committed tokens, grey for guesses, green/red/blue for accepted/rejected/corrected, matching Figure 1 so the demo and the paper read as one artifact.


## How we built the speculative decoding visualizer

The app animates the paper's Figure 1: a small model proposes tokens, the large model verifies them in one parallel run, and the viewer watches speculative decoding pull ahead of standard decoding. It runs in FastHTML inside this Solveit dialog. `JupyUvi` runs the server in a background thread and `HTMX()` shows the app in an iframe, so we could redefine routes and see the result without restarting anything.

We made six decisions worth recording.

We let the server drive the animation rather than writing client-side JS. Each beat is an HTMX request that swaps in the next state. This is idiomatic FastHTML, and it also mirrors the algorithm: one round-trip per run of the target model.

Routes are stateless. The tick number travels in the URL, as in `step?tick=3`, so any request fully describes what to render. Nothing on the server can get out of sync, and the back button came for free.

The data is pre-scripted instead of coming from live models. Each iteration is a dict holding the committed prefix, the guesses, the number accepted, and the correction, with each prefix equal to the previous prefix plus its accepted guesses plus its correction. We also included the guesses that get thrown away after the first rejection. The paper's figure omits them, but they were generated, and showing them makes the wasted work visible.

The colors follow the paper. Committed tokens are black, guesses grey, and accepted, rejected, and corrected tokens are green, red, and blue. The rejected token gets a strikethrough so red and blue can sit side by side.

The two tracks are stacked, standard on top as the baseline, and swap together inside one container so they cannot drift apart. Two ticks make one target-model run. The standard track reveals one token per odd tick at the same pace for the whole animation. It does not speed up after the speculative track finishes, which keeps the comparison honest.

Stepping is manual. We started with a timer, then switched to Back and Next buttons that advance one tick per click, so a viewer can sit on the verification beat as long as they like. A step counter and phase labels ("M_q guessing", "M_p verifying") show where you are and which model is working.

The payoff comes at the end: speculative decoding finishes all 14 tokens in 3 steps, then sits idle while standard decoding plods through its remaining 11.


### Getting the CSS/Style right

Three problems came up getting the token rows to wrap cleanly.

**Tokens overflowing despite `flex-wrap`.** `flex-wrap` on a container does nothing if its children are allowed to declare a minimum width equal to their content — which they are by default (`min-width:auto`). `min-width:0` on the flex item overrides this and lets wrapping actually fire. This is the magic line; nearly every "won't wrap" bug traces back to it.

**Tokens breaking mid-content.** `white-space:nowrap` on each child element keeps it as a discrete unit. The browser can only break *between* elements, never inside them. Apply it to the tokens and to any multi-word labels like "M_p verifying (parallel)". Do not put it on the parent container — that switches wrapping off entirely.

**Rows losing horizontal alignment after wrapping.** Content-sized labels only *look* aligned when they happen to be the same length. Lock every label to the same fixed width with `flex:0 0 12em` (grow 0, shrink 0, fixed basis). Switch the row container to `align-items:flex-start` so a taller wrapped block doesn't drag its label to the vertical middle.


## Coding choices

Let's go with the server-driven HTMX approach then.  I like that it embodies the algorithm and that it is more idiomatic.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great choice — let's design the HTMX flow before writing code.

The core mechanism in HTMX for a timed sequence is a `Div` that requests its own next state from the server, using `hx_trigger` with a delay. So each beat is a server response that swaps in the row's new appearance, and includes the trigger for the *next* beat.

For one iteration, the three server responses would be:

1. **Propose** — render $M_q$'s guesses as grey tokens; include `hx_trigger="load delay:1s"` to fetch beat 2.
2. **Validate** — same tokens but with a "flash" style (signalling $M_p$'s parallel run); trigger beat 3 after a short delay.
3. **Resolve** — recolor to green/red/blue and drop the discarded tokens; then trigger the *next iteration*.

So the row essentially "pulls" itself through the beats, each swap being one server round-trip — which, as we said, mirrors a run of $M_p$.

One design question to settle first: should the server hold the **state** (which iteration/beat we're on) so each request just says "give me the next step"? Or should each request carry the state in its URL/parameters (e.g. `/step?iter=0&beat=1`)?

The second (stateless, params-in-URL) is usually cleaner and easier to reason about. Which do you prefer?

Let's got with stateless, params-in-URL

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good — stateless keeps the logic transparent, and it means any request fully describes what to render, with no hidden server memory to get out of sync.

Before wiring up routes, we need the **data** the routes will read: the pre-scripted sequence of iterations. Since the server is just a "playback" of a fixed run here, each iteration needs to know:

- the **committed prefix** (black tokens carried from before)
- the **guesses** $M_q$ proposed this iteration
- **how many** were accepted before the first rejection
- the **correction** token from $M_p$

A natural shape is a list of iterations, each something like:

```python
{'prefix': [...committed tokens...],
 'guesses': [...γ guessed tokens...],
 'n_accepted': 4,          # accepted before first rejection
 'correction': 'index'}    # M_p's replacement for the rejected guess
```

From that one record, a route can derive every beat's colors: propose = all guesses grey; resolve = first `n_accepted` green, next one red, correction blue, rest dropped.

Does this data shape look right to you? If so, the next small step is for you to write out two or three iterations of the Nikkei sentence in this form, and we'll build the first route against it.

What is up with the .to syntax used by HTMX?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a FastHTML feature, not HTMX itself. When you decorate a function with `@rt`, FastHTML attaches a `.to()` method to it that generates the URL for that route, including query parameters. So:

```python
resolve.to(i=2)
```

produces something like `/resolve?i=2`.

It's FastHTML's equivalent of Flask's `url_for()` — you reference the function directly instead of hardcoding URL strings, so if you rename a route, everything stays in sync.

Now that we've resolved the wrapping issues, can you rewrite all_sentences to start with 3 tokens as prefix for each sentence? Please use variation in the number of correct tokens accepted.  I'd like to see it vary between 0-5, taking on all possible values.  Also, please write them compactly, like in my examples.  My examples had a variety of topics/styles (news, scientific, fairy tale).  Please keep that variety as well. Even though these have different topics, the style of the sentences still feels the same.  I want sentences that feel like they have different stylistic sources: a scientific paper, a children' book, a legal clause, and literary fiction.  Each iteration must have 5 guesses.  I would like variety in the number of iterations it takes to complete the sentence, varying between 3-5

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the rewritten data. Each sentence has a 3-token seed prefix, every iteration has exactly 5 guesses, iteration counts are 3/4/5/4, and `n_accepted` takes every value from 0–5 across the set (scientific: 2,5,0 · children's: 4,1,3,0 · legal: 0,2,1,3,2 · literary: 1,3,2,4):

```python
all_sentences = [
    # Scientific paper — n_accepted: 2, 5, 0  (3 iters)
    ({'prefix': ['Mean','reaction','times'], 'guesses': ['decreased','significantly','with','age','and'], 'n_accepted': 2, 'correction': 'across'},
     {'prefix': ['Mean','reaction','times','decreased','significantly','across'], 'guesses': ['all','four','experimental','conditions','after'], 'n_accepted': 5, 'correction': 'the'},
     {'prefix': ['Mean','reaction','times','decreased','significantly','across','all','four','experimental','conditions','after','the'], 'guesses': ['training','experiment','test','trial','period'], 'n_accepted': 0, 'correction': 'intervention'},
     ['Mean','reaction','times','decreased','significantly','across','all','four','experimental','conditions','after','the','intervention']),

    # Children's book — n_accepted: 4, 1, 3, 0  (4 iters)
    ({'prefix': ['Little','Bear','woke'], 'guesses': ['up','one','sunny','morning','and'], 'n_accepted': 4, 'correction': 'feeling'},
     {'prefix': ['Little','Bear','woke','up','one','sunny','morning','feeling'], 'guesses': ['very','hungry','so','he','went'], 'n_accepted': 1, 'correction': 'grumpy'},
     {'prefix': ['Little','Bear','woke','up','one','sunny','morning','feeling','very','grumpy'], 'guesses': ['because','he','had','not','slept'], 'n_accepted': 3, 'correction': 'no'},
     {'prefix': ['Little','Bear','woke','up','one','sunny','morning','feeling','very','grumpy','because','he','had','no'], 'guesses': ['honey','breakfast','food','porridge','snacks'], 'n_accepted': 0, 'correction': 'berries'},
     ['Little','Bear','woke','up','one','sunny','morning','feeling','very','grumpy','because','he','had','no','berries']),

    # Legal clause — n_accepted: 0, 2, 1, 3, 2  (5 iters)
    ({'prefix': ['The','undersigned','parties'], 'guesses': ['agree','shall','must','will','may'], 'n_accepted': 0, 'correction': 'hereby'},
     {'prefix': ['The','undersigned','parties','hereby'], 'guesses': ['agree','to','the','following','terms'], 'n_accepted': 2, 'correction': 'indemnify'},
     {'prefix': ['The','undersigned','parties','hereby','agree','to','indemnify'], 'guesses': ['the','other','party','against','all'], 'n_accepted': 1, 'correction': 'Company'},
     {'prefix': ['The','undersigned','parties','hereby','agree','to','indemnify','the','Company'], 'guesses': ['against','any','and','every','claim'], 'n_accepted': 3, 'correction': 'all'},
     {'prefix': ['The','undersigned','parties','hereby','agree','to','indemnify','the','Company','against','any','and','all'], 'guesses': ['claims','arising','from','this','contract'], 'n_accepted': 2, 'correction': 'hereunder'},
     ['The','undersigned','parties','hereby','agree','to','indemnify','the','Company','against','any','and','all','claims','arising','hereunder']),

    # Literary fiction — n_accepted: 1, 3, 2, 4  (4 iters)
    ({'prefix': ['The','rain','had'], 'guesses': ['stopped','falling','all','night','long'], 'n_accepted': 1, 'correction': 'hours'},
     {'prefix': ['The','rain','had','stopped','hours'], 'guesses': ['ago','but','the','city','was'], 'n_accepted': 3, 'correction': 'streets'},
     {'prefix': ['The','rain','had','stopped','hours','ago','but','the','streets'], 'guesses': ['were','still','slick','with','water'], 'n_accepted': 2, 'correction': 'silver'},
     {'prefix': ['The','rain','had','stopped','hours','ago','but','the','streets','were','still','silver'], 'guesses': ['with','rain','under','the','moon'], 'n_accepted': 4, 'correction': 'streetlights'},
     ['The','rain','had','stopped','hours','ago','but','the','streets','were','still','silver','with','rain','under','the','streetlights']),
]
```

One thing to be aware of before running: the `n_accepted: 5` case (scientific paper, iteration 2) will crash the current `step` route. In the resolve branch, `g[it['n_accepted']]` does `g[5]` on a 5-element list → `IndexError`. When all 5 guesses are accepted there *is* no rejected token — the correction is the free bonus token $M_p$ appends (the $\gamma+1$ case from the paper). So the resolve branch needs a small tweak, something like:

```python
red = [Tok(g[it['n_accepted']], 'rejected', 'text-decoration:line-through')] if it['n_accepted'] < len(g) else []
spec_row = Div(*prefix, *green, *red, blue, ...)
```

That's also a nice teaching moment in the viz: viewers will see an iteration where nothing is red and the blue token is pure profit.

## Code

In [ ]:
from fasthtml.common import *

In [ ]:
colors = {'accepted':'#22c55e', 'rejected':'#ef4444', 'correction':'#3b82f6', 'committed':'#000000', 'guess':'#9ca3af'}

In [ ]:
all_sentences = [
    # Scientific paper — n_accepted: 2, 5, 0  (3 iters)
    ({'prefix': ['Mean','reaction','times'], 'guesses': ['decreased','significantly','with','age','and'], 'n_accepted': 2, 'correction': 'across'},
     {'prefix': ['Mean','reaction','times','decreased','significantly','across'], 'guesses': ['all','four','experimental','conditions','after'], 'n_accepted': 5, 'correction': 'the'},
     {'prefix': ['Mean','reaction','times','decreased','significantly','across','all','four','experimental','conditions','after','the'], 'guesses': ['training','experiment','test','trial','period'], 'n_accepted': 0, 'correction': 'intervention'},
     ['Mean','reaction','times','decreased','significantly','across','all','four','experimental','conditions','after','the','intervention']),

    # Children's book — n_accepted: 4, 1, 3, 1  (3 iters)
    ({'prefix': ['Little','Bear','woke'], 'guesses': ['up','one','sunny','morning','and'], 'n_accepted': 4, 'correction': 'feeling'},
     {'prefix': ['Little','Bear','woke','up','one','sunny','morning','feeling'], 'guesses': ['very','hungry','so','he','went'], 'n_accepted': 1, 'correction': 'grumpy'},
     {'prefix': ['Little','Bear','woke','up','one','sunny','morning','feeling','very','grumpy'], 'guesses': ['because','he','had','not','slept'], 'n_accepted': 3, 'correction': 'no'},
     {'prefix': ['Little','Bear','woke','up','one','sunny','morning','feeling','very','grumpy','because','he','had','no'], 'guesses': ['porridge', 'to', 'eat', 'for', 'his'], 'n_accepted': 1, 'correction': '.'},
     ['Little','Bear','woke','up','one','sunny','morning','feeling','very','grumpy','because','he','had','no','porridge', '.']),

    # Legal clause — n_accepted: 0, 2, 1, 3, 2  (4 iters)
    ({'prefix': ['The','undersigned','parties'], 'guesses': ['agree','shall','must','will','may'], 'n_accepted': 0, 'correction': 'hereby'},
     {'prefix': ['The','undersigned','parties','hereby'], 'guesses': ['agree','to','the','following','terms'], 'n_accepted': 2, 'correction': 'indemnify'},
     {'prefix': ['The','undersigned','parties','hereby','agree','to','indemnify'], 'guesses': ['the','other','party','against','all'], 'n_accepted': 1, 'correction': 'Company'},
     {'prefix': ['The','undersigned','parties','hereby','agree','to','indemnify','the','Company'], 'guesses': ['against','any','and','every','claim'], 'n_accepted': 3, 'correction': 'all'},
     {'prefix': ['The','undersigned','parties','hereby','agree','to','indemnify','the','Company','against','any','and','all'], 'guesses': ['claims','arising','from','this','contract'], 'n_accepted': 2, 'correction': 'hereunder'},
     ['The','undersigned','parties','hereby','agree','to','indemnify','the','Company','against','any','and','all','claims','arising','hereunder']),

    # Literary fiction — n_accepted: 1, 3, 2, 4  (4 iters)
    ({'prefix': ['The','rain','had'], 'guesses': ['stopped','falling','all','night','long'], 'n_accepted': 1, 'correction': 'hours'},
     {'prefix': ['The','rain','had','stopped','hours'], 'guesses': ['ago','but','the','city','was'], 'n_accepted': 3, 'correction': 'streets'},
     {'prefix': ['The','rain','had','stopped','hours','ago','but','the','streets'], 'guesses': ['were','still','slick','with','water'], 'n_accepted': 2, 'correction': 'silver'},
     {'prefix': ['The','rain','had','stopped','hours','ago','but','the','streets','were','still','silver'], 'guesses': ['with','rain','under','the','moon'], 'n_accepted': 4, 'correction': 'streetlights'},
     ['The','rain','had','stopped','hours','ago','but','the','streets','were','still','silver','with','rain','under','the','streetlights']),
]


In [ ]:
# all_sentences = [
#     ({'prefix': [], 'guesses': ['Global', 'stocks', 'fell', 'sharply', 'after'],
#      'n_accepted': 1, 'correction': 'markets'},
#     {'prefix': ['Global', 'markets'],
#      'guesses': ['surged', 'after', 'the', 'central', 'government', 'decided'],
#      'n_accepted': 4, 'correction': 'bank'},
#     {'prefix': ['Global', 'markets', 'surged', 'after', 'the', 'central', 'bank'],
#      'guesses': ['announced', 'unexpected', 'interest', 'rate', 'changes'],
#      'n_accepted': 2, 'correction': 'rate'},
#     {'prefix': ['Global', 'markets', 'surged', 'after', 'the', 'central', 'bank', 'announced', 'unexpected', 'rate'],
#      'guesses': ['cuts', 'last', 'week', 'following', 'pressure'],
#      'n_accepted': 2, 'correction': 'Friday'},
#     ['Global', 'markets', 'surged', 'after', 'the', 'central', 'bank', 'announced', 'unexpected', 'rate', 'cuts', 'last', 'Friday']),
#     ({'prefix': [], 'guesses': ['The', 'robot', 'quickly', 'started', 'to', 'move'],
#      'n_accepted': 2, 'correction': 'slowly'},
#     {'prefix': ['The', 'robot', 'slowly'],
#      'guesses': ['learned', 'to', 'adapt', 'through', 'trial'],
#      'n_accepted': 2, 'correction': 'paint'},
#     {'prefix': ['The', 'robot', 'slowly', 'learned', 'to', 'paint'],
#      'guesses': ['pictures', 'of', 'the', 'sea', 'and', 'sky'],
#      'n_accepted': 3, 'correction': 'ocean'},
#     ['The', 'robot', 'slowly', 'learned', 'to', 'paint', 'pictures', 'of', 'the', 'ocean']),  
#     ({'prefix': [], 'guesses': ['The', 'new', 'telescope', 'found'],
#      'n_accepted': 1, 'correction': 'satellite'},
#     {'prefix': ['The', 'satellite'],
#      'guesses': ['captured', 'detailed', 'images', 'of', 'the', 'surface'],
#      'n_accepted': 4, 'correction': 'volcanic'},
#     {'prefix': ['The', 'satellite', 'captured', 'detailed', 'images', 'of', 'volcanic'],
#      'guesses': ['activity', 'deep', 'below', 'ground'],
#      'n_accepted': 2, 'correction': 'beneath'},
#     {'prefix': ['The', 'satellite', 'captured', 'detailed', 'images', 'of', 'volcanic', 'activity', 'deep', 'beneath'],
#      'guesses': ['the', 'southern', 'polar', 'cap', 'region'],
#      'n_accepted': 2, 'correction': 'ice'},
#     {'prefix': ['The', 'satellite', 'captured', 'detailed', 'images', 'of', 'volcanic', 'activity', 'deep', 'beneath', 'the', 'southern', 'ice'],
#      'guesses': ['caps', 'melting', 'rapidly'],
#      'n_accepted': 0, 'correction': 'shelf'},
#     ['The', 'satellite', 'captured', 'detailed', 'images', 'of', 'volcanic', 'activity', 'deep', 'beneath', 'the', 'southern', 'ice', 'shelf']),
#     ({'prefix': [], 'guesses': ['Once', 'upon', 'a', 'time', 'there', 'was'],
#      'n_accepted': 5, 'correction': 'lived'},
#     {'prefix': ['Once', 'upon', 'a', 'time', 'there', 'lived'],
#      'guesses': ['a', 'brave', 'little', 'girl', 'who', 'loved'],
#      'n_accepted': 3, 'correction': 'rabbit'},
#     {'prefix': ['Once', 'upon', 'a', 'time', 'there', 'lived', 'a', 'brave', 'little', 'rabbit'],
#      'guesses': ['in', 'a', 'cozy', 'hole', 'under'],
#      'n_accepted': 3, 'correction': 'burrow'},
#     ['Once', 'upon', 'a', 'time', 'there', 'lived', 'a', 'brave', 'little', 'rabbit', 'in', 'a', 'cozy', 'burrow'])
# ]

### App: Version 3 (Standard version + Speculative Decoding + Multiple Sentences)

In [ ]:
# the app only needs to be created once
app, rt = fast_app(pico=True)

In [ ]:
def Tok(text, status, extra=''):
    return Span(text, style=f'background:{colors[status]};color:white;padding:2px 6px;margin:2px;border-radius:4px;white-space:nowrap;{extra}')

In [ ]:
def tick_state(tick, iters):
    n_iters = len(iters)
    spec_iter = tick // 2           # which speculative iteration
    is_resolve = tick % 2 == 1      # propose or resolve?
    spec_done = spec_iter >= n_iters
    n_standard = (tick + 1) // 2    # standard reveals one token per odd tick
    return spec_iter, is_resolve, spec_done, n_standard

In [ ]:
def wrap_row(name, row):
    return Div(Strong(name, style='flex:0 0 12em'),
               Div(row, style='flex:1;min-width:0;display:flex;flex-wrap:wrap'),
               style='display:flex;align-items:flex-start')

In [ ]:
@rt
def step(tick: int = 0, sent: int=0):
    seed_length = 3         # number of tokens given as a seed at the start
    *iters, std_tokens = all_sentences[sent]
    spec_iter, is_resolve, spec_done, n_standard = tick_state(tick, iters)
    
    # determine phase for speculative decoding
    if is_resolve:
        phase = Span('M', Sub('p'), ' verifying (parallel)', style='white-space:nowrap') 
    else:
        phase = Span('M', Sub('q'), ' guessing', style='white-space:nowrap')
    if spec_done: phase = Span('finished', style='white-space:nowrap')

    # standard approach (without speculative decoding)
    # if n_standard == 0: std_row = Div(Em('(running…)', style='color:#9ca3af'))
    std_row = Div(*(Tok(t, 'committed') for t in std_tokens[:n_standard+seed_length]), style='min-width:0')

    if spec_done: 
        spec_row = Div(*(Tok(t, 'committed') for t in std_tokens), Span(phase, style='color:#9ca3af;margin-left:1em'), style='min-width:0')
    else:
        it = iters[spec_iter]
        prefix = [Tok(t, 'committed') for t in it['prefix']]
        if is_resolve:
            g = it['guesses']
            green = [Tok(t, 'accepted') for t in g[:it['n_accepted']]]      # accepted           
            red = [Tok(g[it['n_accepted']], 'rejected', 'text-decoration:line-through')] if it['n_accepted'] < len(g) else []
#             red = Tok(g[it['n_accepted']], 'rejected', 'text-decoration:line-through')      # first rejection
            blue = Tok(it['correction'], 'correction')                      # M_p's fix
            # g[n_accepted+1:] are discarded — not rendered
            spec_row = Div(*prefix, *green, red, blue, Span(phase, style='color:#9ca3af;margin-left:1em'), style='min-width:0')
        else:
            guesses = [Tok(t, 'guess') for t in it['guesses']]
            spec_row = Div(*prefix, *guesses, Span(phase, style='color:#9ca3af;margin-left:1em'), style='min-width:0')

    counter = Span(f'Step {tick//2 + 1} of {len(std_tokens) - seed_length}', style='margin:0 1em')
    back_btn = Button('◀ Back', hx_get=step.to(tick=tick-1, sent=sent), hx_target='#tracks', hx_swap='outerHTML', disabled=tick==0)
    next_btn = Button('Next ▶', hx_get=step.to(tick=tick+1, sent=sent), hx_target='#tracks', hx_swap='outerHTML', 
                disabled=tick+1>=2*(len(std_tokens)-seed_length))
    
    change_btn = Button('Change sentence', hx_get=step.to(tick=0, sent=(sent+1) % len(all_sentences)), hx_target='#tracks',    
                hx_swap='outerHTML')   

    return Div(wrap_row('Standard', std_row), Hr(), wrap_row('Speculative Decoding', spec_row), Hr(),
           Div(back_btn, counter, next_btn), Hr(), Div(change_btn), id='tracks', hx_swap='outerHTML')



In [ ]:
@rt
def index(): return Titled('Speculative Decoding Visualization', step(0))